# SHAP Feature Importance Analysis
## Hazara Division Air Quality Intelligence System

This notebook uses SHAP (SHapley Additive exPlanations) to explain the trained AQI prediction model.

**Objectives:**
- Identify the most influential features for AQI prediction
- Understand how each feature impacts predictions
- Visualize feature interactions and dependencies

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from src.preprocessing import preprocess_data

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)

print('Libraries loaded successfully.')

## 1. Load Model and Data

In [ ]:
# Load the trained model
with open('../models/model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('../models/features.pkl', 'rb') as f:
    features = pickle.load(f)

model_name = type(model).__name__
print(f'Model Type: {model_name}')
print(f'Features ({len(features)}): {features}')

In [ ]:
# Load dataset and filter for Abbottabad (primary training district)
df = pd.read_csv('../data/cleaned_hazara_aqi.csv')
df = df[df['district'] == 'Abbottabad'].copy()

# Remove timezone and filter historical data
df['date'] = pd.to_datetime(df['date']).dt.tz_localize(None)
from datetime import datetime
df = df[df['date'] < datetime.now()]

if 'district' in df.columns:
    df = df.drop(columns=['district'])

# Preprocess
df = preprocess_data(df, is_training=True)
df['target'] = df['us_aqi'].shift(-1)
df = df.dropna()

X = df[features]
print(f'Dataset shape: {X.shape}')
X.head()

## 2. Compute SHAP Values

In [ ]:
# Use a sample for faster SHAP computation
sample_size = min(500, len(X))
X_sample = X.sample(n=sample_size, random_state=42)

# Choose explainer based on model type
if model_name in ['XGBRegressor', 'RandomForestRegressor']:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
else:
    # For Linear Regression, use the general Explainer
    explainer = shap.Explainer(model, X_sample)
    shap_values = explainer(X_sample).values

print(f'SHAP values computed for {sample_size} samples.')
print(f'SHAP values shape: {shap_values.shape}')

## 3. Global Feature Importance (Bar Plot)

In [ ]:
# Mean absolute SHAP values
mean_shap = np.abs(shap_values).mean(axis=0)
feature_importance = pd.DataFrame({
    'Feature': features,
    'Mean |SHAP|': mean_shap
}).sort_values('Mean |SHAP|', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(feature_importance)))
ax.barh(
    feature_importance['Feature'],
    feature_importance['Mean |SHAP|'],
    color=colors,
    edgecolor='white',
    alpha=0.85
)
ax.set_title('Feature Importance (Mean |SHAP| Value)', fontweight='bold', fontsize=14)
ax.set_xlabel('Mean |SHAP| Value')
ax.grid(alpha=0.2, axis='x')

plt.tight_layout()
plt.show()

print('\nTop 5 Most Important Features:')
print(feature_importance.tail(5).to_string(index=False))

## 4. SHAP Summary Plot (Beeswarm)

In [ ]:
shap.summary_plot(shap_values, X_sample, feature_names=features, show=True)

## 5. Feature Dependence Plots

These plots show how a single feature's value affects the model's output.

In [ ]:
# Plot dependence for the top 3 features
top_features = feature_importance.tail(3)['Feature'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for i, feat in enumerate(top_features):
    feat_idx = features.index(feat)
    shap.dependence_plot(
        feat_idx, shap_values, X_sample,
        feature_names=features,
        ax=axes[i],
        show=False
    )
    axes[i].set_title(f'SHAP Dependence: {feat}', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Interpretation

### Expected Key Findings:

1. **`lag_1h_aqi`**: The most important feature, representing the strong persistence of air quality.
2. **`us_aqi_rolling_24h`**: Daily average trend acts as a strong predictor.
3. **`pm2_5_rolling_24h`**: Fine particulate matter concentration drives AQI.
4. **`lag_24h_aqi`**: Capture daily cycle dependencies.
5. **`hour`**: Time of day captures traffic and daily cycle patterns.

### What This Means for Hazara Division:
- Air quality is persistent; recent history is the best predictor.
- PM2.5 from wood combustion, traffic, and seasonal changes is likely the main pollutant.
- Narrow valleys and mountainous terrain can trap pollutants, especially during winter temperature inversions.